In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/size_only/benchmarks/scaling'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# DeepRV Scaling Curve

This notebook compares DeepRV GPU throughput with GPflow throughput using 48 CPU cores as the initial population size increases.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
results_file = GENERATED / 'soay/size_only/benchmarks/scaling/scaling_summary.json'
output_directory = GENERATED / 'soay/size_only/benchmarks/scaling'

with results_file.open() as file:
    summary = json.load(file)

results = pd.DataFrame({
    "Initial Population Size": [row["initial_population_size"] for row in summary["scenarios"]],
    "DeepRV Simulation Seconds": [row["deeprv"]["simulation_seconds"] for row in summary["scenarios"]],
    "GPflow Simulation Seconds": [row["gpflow"]["simulation_seconds"] for row in summary["scenarios"]],
    "DeepRV Datasets Per Second": [row["deeprv"]["datasets_per_second"] for row in summary["scenarios"]],
    "GPflow Datasets Per Second": [row["gpflow"]["datasets_per_second"] for row in summary["scenarios"]],
    "DeepRV Speedup": [row["deeprv_speedup_over_parallel_gpflow"] for row in summary["scenarios"]],
}).sort_values("Initial Population Size")

results.round(2)

In [ ]:
plt.style.use("seaborn-v0_8-white")
fig, ax = plt.subplots(figsize=(5.906, 3.65))

ax.plot(results["Initial Population Size"], results["DeepRV Datasets Per Second"], color="#D55E00", marker="o", linewidth=1.5, label="DeepRV")
ax.plot(results["Initial Population Size"], results["GPflow Datasets Per Second"], color="#0072B2", marker="s", linewidth=1.5, label="GPflow")
ax.set_xlabel("Initial Population Size", fontsize=10)
ax.set_ylabel("Datasets Per Second", fontsize=10)
ax.set_xticks(results["Initial Population Size"])
ax.tick_params(axis="both", labelsize=9, direction="out")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()

fig.savefig(output_directory / "scaling_curve_datasets_per_second_rtx6000ada.pdf", bbox_inches="tight")
fig.savefig(output_directory / "scaling_curve_datasets_per_second_rtx6000ada.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.style.use("seaborn-v0_8-white")
fig, ax = plt.subplots(figsize=(5.906, 3.65))

ax.plot(results["Initial Population Size"], results["DeepRV Simulation Seconds"], color="#D55E00", marker="o", linewidth=1.5, label="DeepRV")
ax.plot(results["Initial Population Size"], results["GPflow Simulation Seconds"], color="#0072B2", marker="s", linewidth=1.5, label="GPflow")
ax.set_xlabel("Initial Population Size", fontsize=10)
ax.set_ylabel("Simulation Seconds", fontsize=10)
ax.set_xticks(results["Initial Population Size"])
ax.tick_params(axis="both", labelsize=9, direction="out")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()

fig.savefig(output_directory / "scaling_curve_simulation_seconds_rtx6000ada.pdf", bbox_inches="tight")
fig.savefig(output_directory / "scaling_curve_simulation_seconds_rtx6000ada.png", dpi=300, bbox_inches="tight")
plt.show()